# Al: complete electronic-to-ionic workflow

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/otter-hed/otter/blob/main/notebooks/00-otter_intro.ipynb)

Calculate aluminium with Otter's default settings and plot the electronic structure and ionic workflow, using the same figure layout and colours as the [HTML example](https://otter-hed.github.io/otter/gen_examples/plot_al_full_workflow.html). All calculation and plotting code is included below.

## Install Otter

Run once in a fresh Colab runtime.

In [ ]:
%pip install -q otter-hed

In [ ]:
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np

import otter
from otter import PlasmaWorkflowConfig, solve_plasma_workflow
from otter.plotting import grid_figsize, style_context

print(f"Otter {otter.__version__}")

## Input

Aluminium at $\rho=8.1\,\mathrm{g\,cm^{-3}}$ and $T_e=T_i=1\,\mathrm{eV}$. Only the physical inputs are set; all numerical controls use Otter's defaults.

In [ ]:
ELEMENT = "Al"
RHO_G_CC = 8.1
TE_EV = 1.0
TI_EV = 1.0

## Calculate

Run the average atom and QOZ/HNC. Results stay in memory; no precomputed data or repository scripts are needed. The printed wall time covers the calculation only, excluding installation and plotting.

In [ ]:
config = PlasmaWorkflowConfig(
    elements=[ELEMENT],
    rho_g_cc=RHO_G_CC,
    temperature_ev=TE_EV,
    ion_temperature_ev=TI_EV,
)
calculation_started = perf_counter()
result = solve_plasma_workflow(config)
calculation_elapsed = perf_counter() - calculation_started
electronic = result["electronic"]["result"]
ion = result["ion"]

print(f"mu={electronic['mu']:.8f} Ha")
print(f"Zbar (AA)={electronic['zbar']:.8f}, Zbar (QOZ)={ion['zbar']:.8f}")
print(f"HNC residual={ion['hnc_best_residual']:.3e}")
print(f"Calculation wall time: {calculation_elapsed:.2f} s ({calculation_elapsed / 60:.2f} min)")

## Electronic structure

Electronic densities, full/external effective potentials, and full-AA potential components. The ionic density is $n_{\rm ion}$; the pseudoatom and screening densities satisfy $n_{\rm PA}=n_{\rm full}-n_{\rm ext}$ and $n_{\rm scr}=n_{\rm PA}-n_{\rm ion}$.

In [ ]:
r_e = np.asarray(electronic["r"])
e_mask = r_e <= 8.0
shell = 4.0 * np.pi * r_e**2
r_ws = float(electronic["r_ws"])
temperature = (
    rf"$T_e=T_i={config.temperature_ev:g}$ eV"
    if config.temperature_ev == config.ion_temperature_ev
    else rf"$T_e={config.temperature_ev:g}$ eV, $T_i={config.ion_temperature_ev:g}$ eV"
)
state_title = rf"{config.elements[0]}, $\rho={config.rho_g_cc:g}$ g cm$^{{-3}}$, " + temperature

with style_context("thesis", palette="bing"):
    fig_electronic, (ax_density, ax_potential, ax_components) = plt.subplots(
        1, 3, figsize=grid_figsize(1, 3)
    )
    for key, label in (
        ("n_full", r"$n^{\rm full}$"), ("n_ion", r"$n^{\rm ion}$"),
        ("n_ext", r"$n^{\rm ext}$"), ("n_pa", r"$n^{\rm PA}$"),
        ("n_scr", r"$n^{\rm scr}$"), ("n0", r"$n_0$"),
    ):
        ax_density.plot(r_e[e_mask], (shell * electronic[key])[e_mask], label=label)
    ax_density.axvline(r_ws, color="0.25", ls=":", lw=1.1, label=r"$R_{\rm WS}$")
    ax_density.set(
        xlabel=r"$r$ [Bohr]", ylabel=r"$4\pi r^2n(r)$ [Bohr$^{-1}$]",
        xlim=(-0.5, 8.0), ylim=(-1.0, 15.0), title="Electronic densities",
    )
    ax_density.legend(ncol=2)

    for key, label in (
        ("v_full", r"$V_{\rm eff}^{\rm full}$"), ("v_ext", r"$V_{\rm eff}^{\rm ext}$"),
    ):
        ax_potential.plot(r_e[e_mask], electronic[key][e_mask], lw=2.0, label=label)
    ax_potential.set(
        xlabel=r"$r$ [Bohr]", ylabel=r"$V(r)$ [Ha]",
        xlim=(-0.5, 5.0), ylim=(-1.0, 1.0), title="Effective potentials",
    )

    for key, label, linestyle, width in (
        ("v_full", r"$V_{\rm eff}^{\rm full}$", "-", 2.0),
        ("v_H", r"$V_{\rm H}$", "--", 1.5),
        ("v_xc", r"$V_{\rm xc}$", "--", 1.5),
        ("v_nuc", r"$V_{\rm nuc}$", "--", 1.5),
    ):
        ax_components.plot(
            r_e[e_mask], electronic[key][e_mask], ls=linestyle, lw=width, label=label
        )
    ax_components.set(
        xlabel=r"$r$ [Bohr]", ylabel=r"$V(r)$ [Ha]",
        xlim=(-0.5, 8.0), ylim=(-8.0, 8.0), title="Full-AA potential components",
    )
    for ax in (ax_potential, ax_components):
        ax.axhline(0.0, color="0.5", ls=":", lw=0.9)
        ax.axvline(r_ws, color="0.25", ls=":", lw=1.1, label=r"$R_{\rm WS}$")
    ax_potential.legend()
    ax_components.legend(ncol=2)

    fig_electronic.suptitle(
        state_title + rf", $\mu={electronic['mu']:.5f}$ Ha", y=0.99
    )
    fig_electronic.tight_layout(rect=(0.0, 0.0, 1.0, 0.965))
plt.show()

## Pseudoatom to ion structure

$f(k)$, $q(k)$, $V_{ii}(k)$, $V_{ii}(r)$, $g_{ii}(r)$, and $S_{ii}(k)$ are taken directly from Otter's ionic result. Rerunning either plotting cell does not repeat the calculation.

In [ ]:
k = np.asarray(ion["k"])
r = np.asarray(ion["r"])
k_mask = k <= 8.0
r_mask = r <= 12.0

with style_context("thesis", palette="bing"):
    fig_ionic, axes = plt.subplots(2, 3, figsize=grid_figsize(2, 3))
    ax_f, ax_q, ax_vk, ax_vr, ax_g, ax_s = axes.ravel()

    ax_f.plot(k[k_mask], ion["f_k"][k_mask])
    ax_f.set(title=r"$f(k)=n_{\rm ion}(k)$", xlabel=r"$k$ [Bohr$^{-1}$]", ylabel="electrons")

    ax_q.plot(k[k_mask], ion["q_k"][k_mask], label=r"$q_{\rm used}$")
    ax_q.set(title=r"$q(k)=n_{\rm scr}(k)$", xlabel=r"$k$ [Bohr$^{-1}$]", ylabel="electrons")
    ax_q.legend()

    ax_vk.plot(k[k_mask], ion["vii_k"][k_mask])
    ax_vk.set(title=r"$V_{ii}(k)$", xlabel=r"$k$ [Bohr$^{-1}$]", ylabel=r"Ha Bohr$^3$")

    ax_vr.plot(r[r_mask], ion["vii_r"][r_mask])
    ax_vr.set(title=r"$V_{ii}(r)$", xlabel=r"$r$ [Bohr]", ylabel="Ha", xlim=(-0.5, 12.0))

    ax_g.plot(r[r_mask], ion["gii_r"][r_mask])
    ax_g.axhline(1.0, color="0.5", lw=0.8, ls=":")
    ax_g.set(title=r"$g_{ii}(r)$", xlabel=r"$r$ [Bohr]", ylabel=r"$g_{ii}(r)$", xlim=(-0.5, 12.0))

    ax_s.plot(k[k_mask], ion["sii_k"][k_mask])
    ax_s.axhline(1.0, color="0.5", lw=0.8, ls=":")
    ax_s.set(title=r"$S_{ii}(k)$", xlabel=r"$k$ [Bohr$^{-1}$]", ylabel=r"$S_{ii}(k)$")

    fig_ionic.suptitle(
        rf"{config.elements[0]} pseudoatom/QOZ/HNC, $\rho={config.rho_g_cc:g}$ g cm$^{{-3}}$, "
        + temperature, y=0.99,
    )
    fig_ionic.tight_layout(rect=(0.0, 0.0, 1.0, 0.965))
plt.show()